In [0]:
from pyspark.sql import functions as F

# Load Silver data
df_silver = spark.table("silver.pcos_cleaned")

# Select ONLY the approved feature + label columns
df_gold = df_silver.select(
    F.col("age").cast("int"),
    F.col("bmi").cast("float"),
    F.col("is_menstrual_irregular").cast("boolean"),
    F.col("testosterone_level").cast("float"),
    F.col("follicle_count").cast("int"),
    F.col("has_pcos").cast("boolean")
)

# Enforce basic semantic validity (light checks, not heavy cleaning)
df_gold = df_gold.filter(
    (F.col("age") > 0) &
    (F.col("bmi") > 0) &
    (F.col("testosterone_level") >= 0) &
    (F.col("follicle_count") >= 0)
)

# Final null check (Gold should never contain nulls)
df_gold = df_gold.dropna()

# Write to Gold layer
(
    df_gold.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("gold.pcos_features")
)

print(" Gold table 'gold.pcos_features' successfully created!")

# Display new dataframe
display(spark.table("gold.pcos_features").limit(5))


 Gold table 'gold.pcos_features' successfully created!


age,bmi,is_menstrual_irregular,testosterone_level,follicle_count,has_pcos
24,34.7,true,25.2,20,false
37,26.4,false,57.1,25,false
32,23.6,false,92.7,28,false
28,28.8,false,63.1,26,false
25,22.1,true,59.8,8,false
